# Haiyan in nighttime lights
**Samar–Leyte · Tacloban · four 60-day periods**

Standalone visualization for Rene L. Principe Jr.’s RQ2 / 3MT. Run from top to bottom in the project’s geospatial Python environment. Edit `PROJECT_DIR` if needed.

Uses every valid land pixel in the five Samar–Leyte provinces, with `DNB_BRDF_Corrected_NTL` and `Mandatory_Quality_Flag == 0`. No GHSL mask, brightness floor, coverage threshold, recovery metrics, or observation-support panels. Each map is a pixelwise median; `log1p` changes the display only. Missing values remain missing and are not converted to zero.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # Registers the .rio accessor.
import geopandas as gpd
from rasterio.features import rasterize
from shapely.geometry import box
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# If needed, install in this notebook's kernel:
# %pip install numpy pandas xarray rioxarray geopandas rasterio plotly "dask[array]" "zarr<3"
# Optional PNG export: %pip install kaleido

## 1. Paths and periods
Paths follow Notebook 3 / 03b / 04c. Roads use Notebook 3’s file discovery because its directory was not recorded. Set `ROADS_PATH` directly if more than one road layer is found. Period labels use consecutive 60-day blocks, not calendar months.

In [ ]:
PROJECT_DIR = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)
DATA_DIR = PROJECT_DIR / "datasets"
ZARR_PATH = DATA_DIR / "VNP46/processed/Haiyan_VNP46A2.zarr"
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
TRACK_PATH = DATA_DIR / "yolanda-path-line-/Yolanda Path Line.shp"
ROADS_PATH = None  # Or Path("/full/path/to/Roads.shp").
OUTPUT_DIR = PROJECT_DIR / "output/haiyan_ntl_visual"

if ROADS_PATH is None:
    road_files = sorted(DATA_DIR.glob("**/*[Rr]oad*.shp"))
    if len(road_files) != 1:
        raise FileNotFoundError(f"Set ROADS_PATH to your road layer. Found: {road_files}")
    ROADS_PATH = road_files[0]
for path in [ZARR_PATH, MUNICIPALITIES_PATH, ROADS_PATH, TRACK_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

PROVINCE_COLUMN = "PROVINCE"
PROVINCES = ["Samar", "Eastern Samar", "Northern Samar", "Leyte", "Southern Leyte"]
EVENT_DATE = pd.Timestamp("2013-11-08")
STAGE_WINDOWS = {
    "Before Haiyan": (EVENT_DATE - pd.Timedelta(days=60), EVENT_DATE - pd.Timedelta(days=1)),
    "0–2 months": (EVENT_DATE, EVENT_DATE + pd.Timedelta(days=59)),
    "2–4 months": (EVENT_DATE + pd.Timedelta(days=60), EVENT_DATE + pd.Timedelta(days=119)),
    "4–6 months": (EVENT_DATE + pd.Timedelta(days=120), EVENT_DATE + pd.Timedelta(days=179)),
}
# West, south, east, north; includes downtown, airport and adjacent neighbourhoods.
TACLOBAN_BOUNDS = (124.94, 11.18, 125.06, 11.30)
COLOR_PERCENTILE = 99.5  # Shared display ceiling; does not alter the composites.
for label, (start, end) in STAGE_WINDOWS.items():
    print(f"{label}: {start:%d %b %Y} – {end:%d %b %Y}")

## 2. Load the existing data
Keep the processed Zarr’s radiance values. The original preprocessing copied floating-point GeoTIFF values; do not apply an additional automatic ×0.1 conversion. The map therefore labels brightness qualitatively, and hover values use stored DNB-BRDF units.

In [ ]:
municipalities = gpd.read_file(MUNICIPALITIES_PATH)
if municipalities.crs is None:
    raise ValueError("Municipality boundaries need a defined CRS.")
region = municipalities.loc[
    municipalities[PROVINCE_COLUMN].str.strip().str.title().isin(PROVINCES)
].to_crs("EPSG:4326").copy()
if not set(PROVINCES).issubset(set(region[PROVINCE_COLUMN].str.strip().str.title())):
    raise ValueError("Check PROVINCE_COLUMN and province names: all five provinces are required.")
region.geometry = region.geometry.make_valid()
region = region.loc[region.geometry.notna() & ~region.geometry.is_empty].copy()

roads = gpd.read_file(ROADS_PATH).to_crs("EPSG:4326")
track = gpd.read_file(TRACK_PATH).to_crs("EPSG:4326")
roads = roads.loc[roads.geometry.notna() & ~roads.geometry.is_empty].copy()
track = track.loc[track.geometry.notna() & ~track.geometry.is_empty].copy()
tacloban_roads = gpd.clip(roads, box(*TACLOBAN_BOUNDS))

ntl = xr.open_zarr(ZARR_PATH, consolidated=None, chunks="auto", mask_and_scale=True)
ntl = ntl.set_coords("date")
if ntl.date.dims[0] != "date":
    ntl = ntl.swap_dims({ntl.date.dims[0]: "date"})
ntl = ntl.assign_coords(date=pd.to_datetime(ntl.date.values).normalize()).sortby("date")
if not ntl.indexes["date"].is_unique:
    raise ValueError("Resolve duplicate dates in the Zarr before compositing.")
ntl = ntl.sel(date=slice(STAGE_WINDOWS["Before Haiyan"][0], STAGE_WINDOWS["4–6 months"][1]))
if "processed" in ntl and not bool((ntl.processed == 1).all().compute()):
    raise ValueError("Finish preprocessing the requested dates before drawing the maps.")

stored_crs = ntl.rio.crs or ntl.attrs.get("crs_wkt")
if stored_crs is None and "spatial_ref" in ntl:
    stored_crs = ntl.spatial_ref.attrs.get("crs_wkt")
# EPSG:4326 is documented in the saved project notebooks.
ntl = ntl.rio.write_crs(stored_crs or "EPSG:4326")
if ntl.rio.crs.to_epsg() != 4326:
    raise ValueError("This notebook expects the project's EPSG:4326 processed grid.")

west, south, east, north = region.total_bounds
raster_west, raster_south, raster_east, raster_north = ntl.rio.bounds()

dx, dy = np.abs(ntl.rio.resolution())

xi = np.flatnonzero(
    (ntl.x.values >= west - dx) & (ntl.x.values <= east + dx)
)
yi = np.flatnonzero(
    (ntl.y.values >= south - dy) & (ntl.y.values <= north + dy)
)

if xi.size == 0 or yi.size == 0:
    raise ValueError("The Zarr and Samar–Leyte boundaries do not overlap.")

if (raster_west > west + dx or raster_south > south + dy
        or raster_east < east - dx or raster_north < north - dy):
    print("The Zarr covers only part of Samar–Leyte; lights appear only where raster data exist.")


ntl = ntl.isel(x=slice(xi.min(), xi.max() + 1), y=slice(yi.min(), yi.max() + 1))
land = xr.DataArray(
    rasterize([(geometry, 1) for geometry in region.geometry],
              out_shape=(ntl.sizes["y"], ntl.sizes["x"]),
              transform=ntl.rio.transform(recalc=True), fill=0,
              all_touched=False, dtype="uint8").astype(bool),
    dims=("y", "x"), coords={"y": ntl.y, "x": ntl.x},
)

dnb = ntl["DNB_BRDF_Corrected_NTL"].astype("float32").transpose("date", "y", "x")
mqf = ntl["Mandatory_Quality_Flag"]
valid_ntl = dnb.where(
    land & (mqf == 0) & np.isfinite(dnb) & (dnb >= 0)
    & ~dnb.isin([65535.0, 6553.5])
)

## 3. Four median composites
All available valid observations contribute. There is no minimum-day or spatial-coverage gate. Missing pixels remain transparent; these descriptive maps alone do not establish recovery.

In [ ]:
stage_maps = {}
for label, (start, end) in STAGE_WINDOWS.items():
    period = valid_ntl.sel(date=slice(start, end))
    if period.sizes["date"] == 0:
        raise ValueError(f"No dates available for {label}.")
    missing_dates = pd.date_range(start, end).difference(pd.DatetimeIndex(period.date.values))
    if len(missing_dates):
        print(f"{label}: {len(missing_dates)} calendar dates absent; using available observations.")
    # Keep the reduction within small spatial tiles to limit memory use.
    stage_maps[label] = (
        period.chunk({"date": -1, "y": 128, "x": 128})
        .median("date", skipna=True).compute()
    )
    print(f"Completed: {label}")

# One common scale for every stage and both map extents.
finite_values = np.concatenate([
    values[np.isfinite(values)]
    for values in [stage.values for stage in stage_maps.values()]
])
if finite_values.size == 0:
    raise ValueError("No valid land-pixel nighttime lights in these periods.")
color_max = max(float(np.percentile(np.log1p(finite_values), COLOR_PERCENTILE)), np.log1p(1.0))

## 4. Map context and labels
Downtown uses Notebook 3’s city-centre coordinate. The airport uses the [CAAP aerodrome reference point](https://vatphil.com/viewchart.php?id=348). Labels provide geographic context only; no land-use classification is inferred from brightness. Roads come from the existing project layer.

In [ ]:
# Flatten multipart lines into one trace per layer.
def line_coordinates(geometries):
    longitude, latitude = [], []
    for geometry in geometries:
        if geometry is None or geometry.is_empty:
            continue
        if geometry.geom_type in ("Polygon", "MultiPolygon"):
            geometry = geometry.boundary
        if hasattr(geometry, "geoms"):
            xx, yy = line_coordinates(geometry.geoms)
            longitude.extend(xx)
            latitude.extend(yy)
        elif geometry.geom_type in ("LineString", "LinearRing"):
            xx, yy = geometry.xy
            longitude.extend(list(xx) + [None])
            latitude.extend(list(yy) + [None])
    return longitude, latitude

boundary_x, boundary_y = line_coordinates(region.geometry)
road_x, road_y = line_coordinates(tacloban_roads.geometry)
track_x, track_y = line_coordinates(track.geometry)
REGIONAL_BOUNDS = (west - 0.05, south - 0.05, east + 0.05, north + 0.05)
landmarks = [
    dict(text="Downtown Tacloban", x=125.0015, y=11.2434, ax=-42, ay=-45),
    dict(text="Daniel Z. Romualdez<br>Airport", x=125.02775, y=11.227597, ax=-48, ay=48),
]
BACKGROUND = "#282828"
LAND_COLOR = "#111113"
TEXT_COLOR = "#F5F5F3"
# Inferno hues, with the black endpoint matched to the dark land.
LIGHT_COLORS = [
    [0.00, LAND_COLOR], [0.10, "#1b0c41"], [0.25, "#550f6d"],
    [0.40, "#88226a"], [0.55, "#bb3754"], [0.70, "#e65c30"],
    [0.82, "#f98e09"], [0.92, "#f9c932"], [1.00, "#fcffa4"],
]

## 5. Samar–Leyte above, Tacloban below
Shared brightness scale; native pixels are not smoothed. Landmark labels appear in the first Tacloban panel to keep the four stages clear. The dashed blue line shows Haiyan’s path as geographic context.

In [ ]:
# ============================================================
# HAIYAN NIGHTTIME LIGHTS — 5 COLUMNS × 2 ROWS
# Before | Impact | 0–2 | 2–4 | 4–6 months
# ============================================================

# Set a number, such as 100.0, to fix the brightest display colour.
# None uses the shared 99.5th-percentile ceiling.
# Higher values reduce colour saturation.
DISPLAY_MAX_NTL = 6

IMPACT_START = pd.Timestamp("2013-11-08")
IMPACT_END = pd.Timestamp("2013-11-14")

impact_days = valid_ntl.sel(date=slice(IMPACT_START, IMPACT_END))
if impact_days.sizes["date"] == 0:
    raise ValueError("No dates are available for the impact week.")

impact_map = (
    impact_days
    .chunk({"date": -1, "y": 128, "x": 128})
    .median("date", skipna=True)
    .compute()
)

plot_windows = {
    "Before Haiyan": STAGE_WINDOWS["Before Haiyan"],
    "Impact": (IMPACT_START, IMPACT_END),
    "0–2 months": STAGE_WINDOWS["0–2 months"],
    "2–4 months": STAGE_WINDOWS["2–4 months"],
    "4–6 months": STAGE_WINDOWS["4–6 months"],
}

plot_maps = {
    "Before Haiyan": stage_maps["Before Haiyan"],
    "Impact": impact_map,
    "0–2 months": stage_maps["0–2 months"],
    "2–4 months": stage_maps["2–4 months"],
    "4–6 months": stage_maps["4–6 months"],
}

# One colour scale across all ten maps.
display_values = np.concatenate([
    np.log1p(values[np.isfinite(values)])
    for stage in plot_maps.values()
    for values in [stage.values]
])
if display_values.size == 0:
    raise ValueError("No valid nighttime-lights values are available.")

automatic_max = max(
    float(np.percentile(display_values, COLOR_PERCENTILE)),
    np.log1p(1.0),
)
display_color_max = (
    automatic_max
    if DISPLAY_MAX_NTL is None
    else np.log1p(DISPLAY_MAX_NTL)
)

fig = make_subplots(
    rows=2,
    cols=5,
    horizontal_spacing=0.012,
    vertical_spacing=0.095,
    row_heights=[0.60, 0.40],
    subplot_titles=[
        f"<b>{label}</b><br>"
        f"<span style='font-size:17px;color:#B8B8B8'>"
        f"{start:%d %b %Y} – {end:%d %b %Y}</span>"
        for label, (start, end) in plot_windows.items()
    ] + [""] * 5,
)

for column, (label, stage) in enumerate(plot_maps.items(), start=1):
    for row, bounds in [(1, REGIONAL_BOUNDS), (2, TACLOBAN_BOUNDS)]:
        x0, y0, x1, y1 = bounds

        panel = stage.where(
            (stage.x >= x0) & (stage.x <= x1)
            & (stage.y >= y0) & (stage.y <= y1),
            drop=True,
        )
        panel_land = land.sel(x=panel.x, y=panel.y)

        fig.add_trace(
            go.Heatmap(
                x=panel.x.values,
                y=panel.y.values,
                z=np.where(panel_land.values, 1.0, np.nan),
                colorscale=[[0, LAND_COLOR], [1, LAND_COLOR]],
                zmin=0,
                zmax=1,
                showscale=False,
                hoverinfo="skip",
                zsmooth=False,
            ),
            row=row,
            col=column,
        )

        fig.add_trace(
            go.Heatmap(
                x=panel.x.values,
                y=panel.y.values,
                z=np.log1p(panel.values),
                customdata=panel.values,
                coloraxis="coloraxis",
                zsmooth=False,
                connectgaps=False,
                hoverongaps=False,
                hovertemplate=(
                    f"<b>{label}</b>"
                    "<br>DNB-BRDF: %{customdata:.2f} (stored units)"
                    "<br>%{y:.4f}° N, %{x:.4f}° E"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=column,
        )

        fig.add_trace(
            go.Scatter(
                x=boundary_x,
                y=boundary_y,
                mode="lines",
                line=dict(
                    color="rgba(150,150,150,0.24)",
                    width=0.45,
                ),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=column,
        )

        if row == 2:
            fig.add_trace(
                go.Scatter(
                    x=road_x,
                    y=road_y,
                    mode="lines",
                    line=dict(
                        color="rgba(208,208,208,0.32)",
                        width=0.55,
                    ),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=row,
                col=column,
            )
        else:
            fig.add_trace(
                go.Scatter(
                    x=track_x,
                    y=track_y,
                    mode="lines",
                    name="Haiyan · 8 Nov 2013",
                    line=dict(
                        color="#82A7C4",
                        width=1.4,
                        dash="dash",
                    ),
                    opacity=0.8,
                    showlegend=(column == 1),
                    hoverinfo="skip",
                ),
                row=row,
                col=column,
            )

            fig.add_shape(
                type="rect",
                x0=TACLOBAN_BOUNDS[0],
                y0=TACLOBAN_BOUNDS[1],
                x1=TACLOBAN_BOUNDS[2],
                y1=TACLOBAN_BOUNDS[3],
                line=dict(
                    color="rgba(255,255,255,0.6)",
                    width=0.8,
                ),
                fillcolor="rgba(0,0,0,0)",
                row=row,
                col=column,
            )

        axis_number = (row - 1) * 5 + column
        xref = "x" if axis_number == 1 else f"x{axis_number}"

        fig.update_xaxes(
            range=[x0, x1],
            visible=False,
            fixedrange=True,
            constrain="domain",
            row=row,
            col=column,
        )
        fig.update_yaxes(
            range=[y0, y1],
            visible=False,
            fixedrange=True,
            scaleanchor=xref,
            scaleratio=1 / np.cos(
                np.deg2rad((y0 + y1) / 2)
            ),
            constrain="domain",
            row=row,
            col=column,
        )

# Landmark labels in the first Tacloban panel.
for landmark in landmarks:
    fig.add_annotation(
        **landmark,
        row=2,
        col=1,
        showarrow=True,
        arrowhead=0,
        arrowwidth=0.8,
        arrowcolor=TEXT_COLOR,
        font=dict(
            family="Arial",
            size=16,
            color=TEXT_COLOR,
        ),
        align="left",
        bgcolor="rgba(25,25,25,0.65)",
        borderpad=3,
    )

fig.update_annotations(
    font=dict(
        family="Arial",
        size=23,
        color=TEXT_COLOR,
    ),
    selector=dict(showarrow=False),
)

fig.update_layout(
    template="plotly_white",
    width=1920,
    height=1080,  # 16:9
    paper_bgcolor=BACKGROUND,
    plot_bgcolor=BACKGROUND,
    font=dict(
        family="Arial",
        size=18,
        color=TEXT_COLOR,
    ),
    title=dict(
        text="<b>HAIYAN | DISASTER IMPACT AND RECOVERY AS SEEN FROM SPACE</b>",
        x=0.045,
        y=0.98,
        font=dict(
            family="Arial",
            size=44,
            color=TEXT_COLOR,
        ),
    ),
    margin=dict(
        l=85,
        r=35,
        t=150,
        b=120,
    ),
    coloraxis=dict(
        colorscale=LIGHT_COLORS,
        cmin=0,
        cmax=display_color_max,
        colorbar=dict(
            title=dict(
                text="Satellite-derived Nighttime lights",
                side="top",
                font=dict(size=24),
            ),
            orientation="h",
            x=0.80,
            xanchor="center",
            y=-0.1,
            yanchor="top",
            len=0.2,
            thickness=20,
            outlinewidth=0,
            tickmode="array",
            tickvals=[0, display_color_max],
            ticktext=["Less", "More"],
            tickfont=dict(
                size=20,
                color=TEXT_COLOR,
            ),
        ),
    ),
    legend=dict(
        x=0.02,
        y=-0.085,
        xanchor="left",
        yanchor="top",
        orientation="h",
        font=dict(size=20),
        bgcolor="rgba(0,0,0,0)",
    ),
)

for row, label in [
    (1, "S A M A R – L E Y T E"),
    (2, "T A C L O B A N"),
]:
    axis = fig.layout.yaxis if row == 1 else fig.layout.yaxis6

    fig.add_annotation(
        x=-0.025,
        y=sum(axis.domain) / 2,
        xref="paper",
        yref="paper",
        text=label,
        textangle=-90,
        showarrow=False,
        font=dict(
            family="Arial",
            size=20,
            color="#CCCCCC",
        ),
    )

fig.show(config={"displayModeBar": False, "responsive": True})

## 6. Save the figure
HTML is self-contained and opens offline. Set `EXPORT_PNG = True` for a high-resolution 3MT image; Kaleido and a compatible Chrome installation are required. The PNG uses an opaque charcoal background to preserve the reference aesthetic on slides.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
html_path = OUTPUT_DIR / "haiyan_ntl_four_stages.html"
fig.write_html(html_path, include_plotlyjs=True,
               config={"displayModeBar": False, "responsive": True})
print(html_path)

EXPORT_PNG = False
if EXPORT_PNG:
    png_path = OUTPUT_DIR / "haiyan_ntl_four_stages.png"
    fig.write_image(png_path, width=1680, height=1080, scale=2)
    print(png_path)